# 01 생성 — HumanEval 코드 생성 (GPU)

**하는 일**: 로컬 모델로 HumanEval 코드를 생성하고, 정지·추출·종료 사유 판정을 **이 노트북에서 한 번만** 해서 저장합니다. 02(평가)와 03(종합)은 여기서 저장한 `exec_code`/`judge_code`를 받아 쓰기만 합니다.

**런타임**: GPU(T4). API 키 불필요.

**실행 순서**
1. `RUN_NAME = "smoke"`로 전체 실행(문제 5개 × 키 3개, 약 10분) → 마지막 셀의 점검표 확인
2. 문제 없으면 `RUN_NAME = "full"`로 바꿔 처음부터 다시 실행(164문제)
3. 세션이 끊기면 그냥 처음부터 다시 실행. 키별로 저장된 문제는 건너뛰고 이어서 생성합니다.

**저장 위치**: `MyDrive/code_eval_v4/<RUN_NAME>/`
- `generations/<model_key>.jsonl` — 문제 1개당 1줄
- `run_meta_generate.json` — 라이브러리 버전, GPU, 모델 revision, 생성 설정

**레코드 필드**: `task_id, model_key, model_id, mode, prompt_version, prompt_sent, raw_response`(자르기 전 원문), `completion`(raw 모드: 정지 문자열에서 자른 본문), `judge_code`(judge가 보는 함수 코드), `exec_code`(pass@1이 실행하는 코드), `extraction_ok, syntax_ok, finish_reason`(stop / eos / length), `n_new_tokens, max_new_tokens, gen_seconds`

**알려진 한계 (보고서에 기재)**
- raw 모드 정지 문자열(`\ndef ` 등)은 Codex 논문 관례를 따름. 본 함수 뒤에 최상위 보조 함수를 정의하면 잘림(이슈로그 A-4) → `finish_reason`과 03의 실패 유형으로 건수 보고
- Qwen(chat: 지시문 + 함수 전체 생성 + 코드블록 추출)과 베이스 모델(raw: 본문 이어쓰기)은 생성 프로토콜이 다름 → 모델 간 순위는 참고용, 이 실험의 주 질문은 judge 검증

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets

In [ ]:
# ============================================================
# 설정 — 여기만 바꾸면 됩니다
# ============================================================
RUN_NAME = "smoke"        # "smoke"(문제 5개로 점검) → 확인 후 "full"(164문제)
N_PROBLEMS = {"smoke": 5, "full": None}[RUN_NAME]

# 결과 키 -> (HF 모델 ID, 모드). raw = 프롬프트 이어쓰기, chat = 지시문 + 함수 전체 생성
GEN_JOBS = {
    "starcoder2-3b":       ("bigcode/starcoder2-3b", "raw"),
    "codellama-7b":        ("codellama/CodeLlama-7b-hf", "raw"),
    "qwen2.5-7b-instruct": ("Qwen/Qwen2.5-7B-Instruct", "chat"),
    # 선택(사이드 노트용): instruct 모델에 raw 프롬프트를 넣은 조건. 필요하면 주석 해제
    # "qwen2.5-7b-instruct-raw": ("Qwen/Qwen2.5-7B-Instruct", "raw"),
}

# 토큰 한도: 두 모드 동일. 정상 생성은 정지 조건에서 먼저 끝나고, 한도까지 가는 건 반복 루프뿐.
# HumanEval 정답 코드는 가장 긴 것도 약 860자(수백 토큰)라 2048이면 정상 코드가 잘릴 일은 사실상 없음.
# 그래도 한도에 닿으면 finish_reason="length"로 기록됨.
MAX_NEW_TOKENS = {"raw": 2048, "chat": 2048}

# ============================================================
import os, json
from google.colab import drive, userdata

drive.mount("/content/drive")
BASE_DIR = "/content/drive/MyDrive/code_eval_v4"
RUN_DIR = f"{BASE_DIR}/{RUN_NAME}"
GEN_DIR = f"{RUN_DIR}/generations"
os.makedirs(GEN_DIR, exist_ok=True)

try:
    HF_TOKEN = userdata.get("HF_TOKEN")      # 없어도 됨(공개 모델). 있으면 다운로드 한도가 넉넉함
except Exception:
    HF_TOKEN = None
print("RUN_DIR:", RUN_DIR, "| 문제 수:", N_PROBLEMS or 164, "| HF 토큰:", bool(HF_TOKEN))

In [ ]:
# 후처리(정지 문자열 자르기, 코드블록 추출, 종료 사유 판정) — 순수 함수, GPU 불필요
# 02(평가)는 이 결과를 그대로 받아 쓰므로, 추출 로직은 프로젝트 전체에서 여기 한 곳에만 있음
import ast, re

STOP_SEQS = ["\ndef ", "\nclass ", "\nif __name__", "\nprint("]   # Codex 논문 관례
PROMPT_VERSION = {"raw": "raw-v1", "chat": "chat-v2"}
CHAT_INSTRUCTION = (
    "Complete the following Python function. "
    "Return the complete function (including the signature) in a single ```python code block. "
    "Do not add any explanation outside the code block.\n\n"
)
FENCE_RE = re.compile(r"```[ \t]*(?:python|py|Python)?[ \t]*\n(.*?)(?:```|\Z)", re.S)

def first_stop(text, stops=STOP_SEQS):
    idx = [text.find(s) for s in stops if s in text]
    return min(idx) if idx else -1

def fence_closed(text):
    return text.count("```") >= 2

def extract_code_block(text):
    """첫 번째 ```python 블록 내용. 펜스가 없으면 (원문, False)"""
    m = FENCE_RE.search(text)
    if m:
        return m.group(1).rstrip() + "\n", True
    return text.strip() + "\n", False

def syntax_ok(code):
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False

def postprocess(mode, problem, raw_text, n_new, ended_eos, limit):
    """모델 원문 -> 저장 레코드의 공통 필드. 판정 우선순위: stop > eos > length"""
    prompt, ep = problem["prompt"], problem["entry_point"]
    if mode == "raw":
        cut = first_stop(raw_text)
        completion = raw_text if cut < 0 else raw_text[:cut]
        stopped = cut >= 0
        judge_code = prompt.strip() + completion          # 모델이 실제로 이어 쓴 형태 그대로
        exec_code = judge_code
        extraction_ok = completion.strip() != ""
    else:
        completion = None
        code, fenced = extract_code_block(raw_text)
        stopped = fence_closed(raw_text)
        judge_code = code
        exec_code = prompt + "\n\n" + code                # 프롬프트의 import·보조 함수를 살리고 함수는 재정의
        has_def = re.search(rf"^\s*def\s+{re.escape(ep)}\s*\(", code, re.M) is not None
        extraction_ok = fenced and has_def
    if stopped:
        finish = "stop"
    elif ended_eos:
        finish = "eos"
    elif n_new >= limit:
        finish = "length"
    else:
        finish = "unknown"
    return {
        "completion": completion,
        "judge_code": judge_code,
        "exec_code": exec_code,
        "extraction_ok": extraction_ok,
        "syntax_ok": syntax_ok(exec_code),
        "finish_reason": finish,
    }

In [ ]:
# 모델 로드 / 생성 (GPU)
import gc, time, logging, warnings
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          StoppingCriteria, StoppingCriteriaList)

logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*MatMul8bitLT.*")
QUANT = BitsAndBytesConfig(load_in_8bit=True)      # 세 모델 모두 8-bit (A-2)

def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or None)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=QUANT, device_map="auto", token=HF_TOKEN or None)
    model.eval()
    return tok, model

def eos_ids(tok, model):
    ids = model.generation_config.eos_token_id
    ids = ids if isinstance(ids, list) else [ids]
    ids = set(i for i in ids if i is not None)
    if tok.eos_token_id is not None:
        ids.add(tok.eos_token_id)
    return ids

class StopCriteria(StoppingCriteria):
    """생성 도중 정지 판정. raw: 정지 문자열 등장, chat: 닫는 코드펜스 등장"""
    def __init__(self, tok, n_prompt, mode):
        self.tok, self.n, self.mode = tok, n_prompt, mode
    def __call__(self, input_ids, scores, **kwargs):
        text = self.tok.decode(input_ids[0][self.n:], skip_special_tokens=True)
        hit = first_stop(text) >= 0 if self.mode == "raw" else fence_closed(text)
        return torch.full((input_ids.shape[0],), hit, dtype=torch.bool, device=input_ids.device)

def build_input(tok, mode, prompt):
    if mode == "raw":
        return prompt.strip()               # 끝 개행을 넣으면 함수를 다시 선언함(B-3)
    msgs = [{"role": "user", "content": CHAT_INSTRUCTION + prompt}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_one(tok, model, mode, prompt, limit, eos_set):
    prompt_sent = build_input(tok, mode, prompt)
    # chat 템플릿 문자열에는 특수 토큰이 이미 들어 있으므로 다시 붙이지 않음
    inputs = tok(prompt_sent, return_tensors="pt", add_special_tokens=(mode == "raw")).to(model.device)
    n = inputs["input_ids"].shape[1]
    out = model.generate(
        **inputs, max_new_tokens=limit, do_sample=False,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
        stopping_criteria=StoppingCriteriaList([StopCriteria(tok, n, mode)]))
    new_ids = out[0][n:]
    raw_text = tok.decode(new_ids, skip_special_tokens=True)   # 자르기 전 원문
    ended_eos = len(new_ids) > 0 and int(new_ids[-1]) in eos_set
    return prompt_sent, raw_text, int(len(new_ids)), ended_eos

In [ ]:
# 실행: 키별로 이어서 생성, 문제 1개 끝날 때마다 jsonl에 한 줄 추가(세션이 끊겨도 손실은 최대 1문제)
import sys, platform, datetime
import transformers, bitsandbytes
from datasets import load_dataset
from tqdm.auto import tqdm

problems = list(load_dataset("openai/openai_humaneval", split="test"))
if N_PROBLEMS:
    problems = problems[:N_PROBLEMS]
torch.manual_seed(0)

def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

meta_path = f"{RUN_DIR}/run_meta_generate.json"
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {"models": {}}
meta.update({
    "run_name": RUN_NAME, "n_problems": len(problems),
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gen_config": {"do_sample": False, "quantization": "8-bit", "stop_seqs_raw": STOP_SEQS,
                   "stop_chat": "closing ``` fence", "max_new_tokens": MAX_NEW_TOKENS,
                   "prompt_version": PROMPT_VERSION, "chat_instruction": CHAT_INSTRUCTION,
                   "limit_reason": "정지 조건으로 정상 생성은 먼저 끝남. 한도는 반복 루프 차단용(정답 코드 최장 약 860자)"},
})

by_model = {}
for key, (model_id, mode) in GEN_JOBS.items():
    by_model.setdefault(model_id, []).append((key, mode))

for model_id, jobs in by_model.items():
    todo = {}
    for key, mode in jobs:
        done = {r["task_id"] for r in read_jsonl(f"{GEN_DIR}/{key}.jsonl")}
        todo[key] = [p for p in problems if p["task_id"] not in done]
        print(f"[{key}] 완료 {len(done)} / 남음 {len(todo[key])}")
    if not any(todo.values()):
        continue
    print(f"로딩: {model_id}")
    tok, model = load_model(model_id)
    eos_set = eos_ids(tok, model)
    meta["models"][model_id] = {"revision": getattr(model.config, "_commit_hash", None)}
    json.dump(meta, open(meta_path, "w"), indent=2, ensure_ascii=False)
    for key, mode in jobs:
        limit = MAX_NEW_TOKENS[mode]
        with open(f"{GEN_DIR}/{key}.jsonl", "a") as f:
            for p in tqdm(todo[key], desc=key):
                t0 = time.time()
                prompt_sent, raw_text, n_new, ended_eos = generate_one(tok, model, mode, p["prompt"], limit, eos_set)
                rec = {"task_id": p["task_id"], "model_key": key, "model_id": model_id, "mode": mode,
                       "prompt_version": PROMPT_VERSION[mode], "prompt_sent": prompt_sent,
                       "raw_response": raw_text,
                       **postprocess(mode, p, raw_text, n_new, ended_eos, limit),
                       "n_new_tokens": n_new, "max_new_tokens": limit,
                       "gen_seconds": round(time.time() - t0, 2),
                       "created_at": datetime.datetime.now().isoformat(timespec="seconds")}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()
    del model, tok
    gc.collect(); torch.cuda.empty_cache()
print("생성 종료")

In [ ]:
# 점검표: smoke에서 여기가 정상이면 full로 넘어감
from collections import Counter

for key in GEN_JOBS:
    recs = read_jsonl(f"{GEN_DIR}/{key}.jsonl")
    ids = [r["task_id"] for r in recs]
    dup = len(ids) - len(set(ids))
    print(f"\n=== {key}: {len(recs)}건 (중복 {dup}) ===")
    print("  종료 사유:", dict(Counter(r["finish_reason"] for r in recs)))
    print("  추출 실패:", [r["task_id"] for r in recs if not r["extraction_ok"]])
    print("  문법 오류:", [r["task_id"] for r in recs if not r["syntax_ok"]])
    print("  한도 도달:", [r["task_id"] for r in recs if r["finish_reason"] == "length"])
    secs = [r["gen_seconds"] for r in recs]
    if secs:
        print(f"  문제당 평균 {sum(secs)/len(secs):.1f}초 → 164문제 예상 {sum(secs)/len(secs)*164/60:.0f}분")
        r = recs[0]
        print(f"  --- 샘플 {r['task_id']} judge_code ---\n{r['judge_code'][:600]}")

# 중복이 있으면(같은 셀을 동시에 두 번 돌린 경우 등) 02에서 첫 레코드만 사용됨